# Transaction Pipeline: Cross-Dataset, Cross-Model Evaluation

This notebook compares how well the transaction preprocessing pipeline's shaped data
trains models, across:
- Multiple datasets (different source CSVs)
- Multiple model sizes of the from-scratch tiny transformer (no GPU needed)
- Real pretrained models fine-tuned on GPU (needs Colab's GPU runtime + internet)

Before running: Runtime -> Change runtime type -> GPU (T4 is fine), for the real-model section.

Honesty note on Part 3 (real pretrained models): the sandbox that built this pipeline
has no GPU and hit disk limits trying to even CPU-install torch, so that part has NOT
been execution-verified end-to-end the way the rest of this project was - it's written
carefully against stable, standard Hugging Face APIs, but this Colab run will be the
first real execution. If a cell errors, share the exact error message and it can be
fixed, the same way earlier bugs in this project were found and fixed.

## Setup

This cell will prompt you to upload `txn_preprocessing_pipeline_v14.zip` if it is
not already present in this Colab session (Colab sessions are ephemeral - any
previously uploaded file is gone after a runtime disconnect/reset, which is the
most common cause of a "file not found" error here).

In [ ]:
import os, subprocess, sys

zip_candidates = [f for f in os.listdir(".") if f.startswith("txn_preprocessing_pipeline") and f.endswith(".zip")]

if not zip_candidates:
    print("Pipeline zip not found in this session - select it in the upload dialog below.")
    print("(If you already uploaded it, the runtime may have reset since then - just upload again.)")
    from google.colab import files
    uploaded = files.upload()
    zip_candidates = [f for f in uploaded.keys() if f.endswith(".zip")]
    if not zip_candidates:
        raise FileNotFoundError("No .zip file was uploaded - re-run this cell and select the pipeline zip.")

zip_name = zip_candidates[0]
print(f"Using: {zip_name}")

# Each step below is checked explicitly - shell commands (!unzip, !pip install)
# do NOT stop notebook execution on failure the way Python code does, so a
# silent failure here would otherwise be followed by a false "Setup complete."
result = subprocess.run(["unzip", "-o", "-q", zip_name], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError(f"unzip failed:\n{result.stderr}\nThe uploaded file may be incomplete or corrupted - try re-uploading it.")

if not os.path.isdir("txn_pipeline"):
    raise FileNotFoundError(
        f"Unzip succeeded but no 'txn_pipeline' folder was produced. "
        f"Contents found here: {os.listdir('.')}"
    )

os.chdir("txn_pipeline")

if not os.path.isfile("requirements.txt"):
    raise FileNotFoundError(f"requirements.txt not found in {os.getcwd()} - contents: {os.listdir('.')}")

result = subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt", "-q"],
                         capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError(f"pip install failed:\n{result.stderr}")

sys.path.insert(0, ".")
sys.path.insert(0, "eval_harness")

assert os.path.isdir("pipeline"), "pipeline/ package not found after setup - extracted contents look wrong."
print("Setup complete - working directory:", os.getcwd())

## Part 1: Prepare multiple datasets

Add as many sources as you want to compare. Starts with the two built-in synthetic
providers (different raw schemas) - replace/extend with your own CSVs via
pd.read_csv("your_file.csv").

In [ ]:
from data_synth import generate_provider_a, generate_provider_b
from eval_harness.evaluation_harness import prepare_datasets, run_grid, summarize_grid, plot_comparison_heatmap
import pandas as pd

sources = {
    "Provider A": generate_provider_a(),
    "Provider B": generate_provider_b(),
    # "My real data": pd.read_csv("my_file.csv"),   # add your own here
}

ALL_TASKS = ["categorization", "fraud_flagging", "spend_summarization", "general_behavioral", "query_sql"]
cached_datasets = prepare_datasets(sources, task_types=ALL_TASKS)
print("Prepared:", list(cached_datasets.keys()))

## Part 2: Compare model sizes on each dataset (from-scratch transformer, no GPU needed)

Runs entirely on CPU in seconds - a fast first pass before spending GPU time on real
pretrained models below. Multiple seeds per cell matter: with small datasets,
differences between configs can be noise, not signal - look at the spread, not just
one run's number.

In [ ]:
results = run_grid(
    cached_datasets,
    task_type="categorization",
    seeds=(0, 1, 2),
    epochs=25,
    lr=0.08,
)
results

In [ ]:
summary = summarize_grid(results)
summary

In [ ]:
fig = plot_comparison_heatmap(summary, value_col="val_accuracy_mean")
fig

## Part 3: Fine-tune REAL pretrained models on GPU

Requires GPU runtime + internet.

### 3a. Categorization: real encoder (DistilBERT)

Directly comparable to Part 2's tiny transformer, since both do the same job.

In [ ]:
!pip install transformers datasets accelerate -q

In [ ]:
from external_llm_eval import run_classification_eval

dataset_name = "Provider A"
examples = cached_datasets[dataset_name]["shaped_outputs"]["categorization"]
train_ex = [e for e in examples if e["split"] == "train"]
val_ex = [e for e in examples if e["split"] in ("val", "test")]

cls_results = run_classification_eval(train_ex, val_ex, epochs=3)

### 3b. Natural-language querying: real causal LM (DistilGPT-2 + LoRA)

The important part: correctness is measured by actually EXECUTING the generated SQL
against the same query engine this pipeline uses, and checking the result matches -
not by comparing generated text to reference text.

In [ ]:
!pip install peft -q

In [ ]:
from external_llm_eval import run_sql_generation_eval
from pipeline.query_engine import build_sqlite_db

sql_examples = cached_datasets[dataset_name]["shaped_outputs"]["query_sql"]
sql_train = [e for e in sql_examples if e["split"] == "train"]
sql_val = [e for e in sql_examples if e["split"] in ("val", "test")]

conn = build_sqlite_db(cached_datasets[dataset_name]["clean_df"])
sql_results = run_sql_generation_eval(sql_train, sql_val, conn, epochs=3)

### Inspect failures

If correct_rate is lower than expected: is the model producing invalid SQL
(executable_rate low too), or valid-but-wrong SQL (executable_rate high, correct_rate
low)? These point to very different fixes.

In [ ]:
for f in sql_results["failures"][:5]:
    print(f)
    print()

### 3c. Fraud flagging: real encoder (reuses run_classification_eval)

Structurally the same job as categorization (text in, label out) - just a multi-transaction context window and FRAUD/LEGITIMATE labels instead of categories, so the same function works directly.

In [ ]:
from external_llm_eval import run_classification_eval

fraud_examples = cached_datasets[dataset_name]["shaped_outputs"]["fraud_flagging"]
fraud_train = [e for e in fraud_examples if e["split"] == "train"]
fraud_val = [e for e in fraud_examples if e["split"] in ("val", "test")]

fraud_results = run_classification_eval(fraud_train, fraud_val, epochs=3)

### 3d. Spend summarization: real causal LM (fact-inclusion checked)

Summaries are open-ended prose, so there's no execution check like SQL has. Instead, correctness is measured as a fact-inclusion rate: does the generated summary actually contain the real computed facts (total spend, top category, etc.) it was supposed to report - not a subjective 'does this read well' judgment.

In [ ]:
from external_llm_eval import run_summarization_eval

summ_examples = cached_datasets[dataset_name]["shaped_outputs"]["spend_summarization"]
summ_train = [e for e in summ_examples if e["split"] == "train"]
summ_val = [e for e in summ_examples if e["split"] in ("val", "test")]

summ_results = run_summarization_eval(summ_train, summ_val, epochs=5)

In [ ]:
# inspect a few generated summaries alongside the facts they were supposed to report
for s in summ_results["samples"][:3]:
    print("Q:", s["instruction"])
    print("Generated:", s["generated"])
    print("Facts used:", s["facts_used"])
    print("Fact-inclusion score:", s["fact_inclusion_score"])
    print()

### 3e. General behavioral: real causal LM (perplexity)

This task is plain sequence continuation with no instruction/answer to check for correctness - it exists for continued-pretraining-style exposure to transaction data structure. Evaluation uses standard held-out perplexity: lower means the model finds real transaction sequences more predictable.

In [ ]:
from external_llm_eval import run_general_behavioral_eval

behavioral_examples = cached_datasets[dataset_name]["shaped_outputs"]["general_behavioral"]
behavioral_train = [e for e in behavioral_examples if e["split"] == "train"]
behavioral_val = [e for e in behavioral_examples if e["split"] in ("val", "test")]

behavioral_results = run_general_behavioral_eval(behavioral_train, behavioral_val, epochs=5)

## Part 4: Combined comparison

From-scratch and real-model results side by side.

In [ ]:
comparison_rows = []
best_scratch = summary[summary["dataset"] == dataset_name].iloc[0]
comparison_rows.append({"approach": f"From-scratch ({best_scratch['model_config']})",
    "task": "categorization", "metric": "val_accuracy", "value": best_scratch["val_accuracy_mean"]})
comparison_rows.append({"approach": "DistilBERT (fine-tuned)",
    "task": "categorization", "metric": "val_accuracy", "value": cls_results["accuracy"]})
comparison_rows.append({"approach": "DistilBERT (fine-tuned)",
    "task": "fraud_flagging", "metric": "val_accuracy", "value": fraud_results["accuracy"]})
comparison_rows.append({"approach": "DistilGPT-2 + LoRA (fine-tuned)",
    "task": "query_sql", "metric": "correct_rate (execution-verified)", "value": sql_results["correct_rate"]})
comparison_rows.append({"approach": "DistilGPT-2 + LoRA (fine-tuned)",
    "task": "spend_summarization", "metric": "fact_inclusion_rate", "value": summ_results["avg_fact_inclusion_rate"]})
comparison_rows.append({"approach": "DistilGPT-2 + LoRA (fine-tuned)",
    "task": "general_behavioral", "metric": "perplexity (lower=better)", "value": behavioral_results["perplexity"]})

pd.DataFrame(comparison_rows)